In [ ]:
# Run once if not in Colab
!pip install huggingface_hub
!pip install transformers datasets tokenizers seqeval -q evaluate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you h

In [ ]:
from utilities_2 import preprocess, create_model_init, get_compute_metrics, concatenate_splits
from huggingface_hub import HfApi
import os
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import numpy as np
import gdown
from seqeval.metrics import f1_score

from datasets import get_dataset_config_names, load_dataset, DatasetDict

from collections import defaultdict, Counter

import torch

from seqeval.metrics import classification_report

In [ ]:
#You can mis this
print(torch.cuda.is_available())  # Should return True if CUDA is properly set up.
print(torch.cuda.get_device_name(0))  # Should return the GPU name if detected.

True
Tesla T4


In [ ]:
# get data
model_name="xlm-roberta-large"
language_code="ru"
#data=preprocess(language_code=language_code, model_name=model_name, train=True)
#tokenized_datasets, label_list, label2id, id2label, tokenizer= data
#print(tokenized_datasets["train"][0])

Get data

In [ ]:

os.makedirs("data", exist_ok=True)
# Create the directories if they don't exist
directories = ["data/test", "data/train", "data/val"]

for directory in directories:
  os.makedirs(directory, exist_ok=True)
#test=["1T1VWy77q-bwcpTk-x6z2eUD8BYWLBWjZ"]#, "1yMy1Ypspn26SPGKtdOd0RuCMUKbkM0Ne", "1oLOiziBhBl5nIcSfzAZw9YNmahbppEAE", "1G-w6_bhLNNlVJ9d3yNMjSFM7gpixZorH", "1OU5YvvBHsASiAE61rTlQEeVyFaWtnsjI"]
train=["1yL1u_mJHY-GyQVSjfAsjFyOHJEH1XqTh", "1ibtJczrFF7uSOqT-b9R4Yx-rfAiwJnWi", "11mSNDRuXCb78ay2x5sQYwCb9_ipeXZwh","161gqjsamWyT9d7UJtLnkCJWOQfHTQ-01","1gzbYd7ofSAV4agIP3GHeepO09j2j7c8Q"]
val=["1eK13wGeI8dQ3WyPpoVonOCIpo7hklUbB","1YBZfdVPm5Upak-1ZRQkS4oKR-FN6BX9z", "1uRy5fIWA7xLr2-dROo119fsq1yyQMRZ_","1QSF8vEo2gky-yxw36rv8iTFuf330kpc_","12YJ5hY91psICvCItLmKHprVC89bcbBUO"]
file_names=["lemmatized_bg.txt", "lemmatized_ru.txt", "lemmatized_sl.txt", "lemmatized_sl_cyrilic.txt","lemmatized_uk.txt"]
#for url, output in zip(test, file_names):
#    gdown.download("https://drive.google.com/uc?id="+ url, "data/test/" + output)
for url, output in zip(train, file_names):
    gdown.download("https://drive.google.com/uc?id="+url, "data/train/" + output)
for url, output in zip(val, file_names):
    gdown.download("https://drive.google.com/uc?id="+url, "data/val/" + output)

In [ ]:
# --- Check for GPU availability ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")


CUDA is available. Using GPU: Tesla T4


In [ ]:
try:
    data = preprocess(language_code=language_code, model_name=model_name, train=True)
    tokenized_dataset_ru, label_list, label2id, id2label, tokenizer = data[0], data[1], data[2], data[3], data[4]
    print("Sample from training data:")
    print(tokenized_dataset_ru["train"][0])
    num_labels = len(label_list)
except NameError:
    print("Error: The 'preprocess' function is not defined.")
    print("Please ensure 'utilities.py' is in the same directory or accessible in your Python path,")
    print("and that it contains the 'preprocess' function.")
    # Example placeholder data if preprocess fails - replace with actual loading if needed
    tokenized_dataset_bg = None # Set to None or load dummy data
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"] # Example labels
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for i, label in enumerate(label_list)}
    num_labels = len(label_list)
    tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer separately if needed
    print("\nWARNING: Using placeholder data because 'preprocess' failed.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/7381 [00:00<?, ? examples/s]

Map:   0%|          | 0/5045 [00:00<?, ? examples/s]

Sample from training data:
{'input_ids': [0, 417, 174222, 103, 34800, 56291, 4988, 183, 1488, 227, 151609, 59, 61, 85063, 244, 29524, 3280, 1468, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [-100, 10, 1, -100, 10, -100, 10, 10, -100, -100, 10, -100, 10, 10, -100, -100, -100, 10, -100, -100, -100, -

In [ ]:
tokenized_dataset_uk= preprocess(language_code="uk", model_name=model_name, train=True)[0]
tokenized_dataset_sl= preprocess(language_code="sl", model_name=model_name, train=True)[0]
tokenized_dataset_ru= preprocess(language_code="ru", model_name=model_name, train=True)[0]

Map:   0%|          | 0/2211 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

Map:   0%|          | 0/7512 [00:00<?, ? examples/s]

Map:   0%|          | 0/5186 [00:00<?, ? examples/s]

Map:   0%|          | 0/7381 [00:00<?, ? examples/s]

Map:   0%|          | 0/5045 [00:00<?, ? examples/s]

## Model training on a single language

Build model with Bulgarian data

In [ ]:
new_model_name = f"{model_name}-finetuned-{language_code}"
model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(output_dir= f"{language_code}-ner-model",
    learning_rate= 3e-5,  # Slightly higher learning rate
    per_device_train_batch_size= 16,  # Maintain current batch size
    per_device_eval_batch_size= 16,
    num_train_epochs=5,  # Increase epochs for better learning
    weight_decay= 0.01,
    #max_seq_length= 200,  # To accommodate longer sentences
    warmup_ratio= 0.1,  # 10% warmup for stable training
    lr_scheduler_type= "linear",
    eval_strategy= "epoch",
    save_strategy= "epoch",
    logging_steps= 700,
    save_total_limit= 2,
    load_best_model_at_end= True,
    metric_for_best_model= "f1",
    seed=42,
    report_to="wandb")

trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=tokenized_dataset_bg["train"],
                  eval_dataset=tokenized_dataset_bg["validation"],
                  tokenizer=tokenizer)

<ipython-input-11-c8bd3c5e59b9>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train

In [ ]:
trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: diko4ev (diko4ev-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.217800,0.191996,0.791652,0.677574,0.730184,0.970204
2,0.034200,0.182206,0.769637,0.700090,0.733218,0.970698
3,0.020400,0.184248,0.868450,0.711498,0.782178,0.974554
4,0.013000,0.197636,0.859175,0.706995,0.775692,0.973783
5,0.008300,0.224218,0.863653,0.705494,0.776603,0.973797


TrainOutput(global_step=3665, training_loss=0.05646129337534573, metrics={'train_runtime': 5374.4132, 'train_samples_per_second': 10.899, 'train_steps_per_second': 0.682, 'total_flos': 1.36001680081344e+16, 'train_loss': 0.05646129337534573, 'epoch': 5.0})

Save

In [ ]:
#Save locally
trainer.save_model("./my_ner_model")
tokenizer.save_pretrained("./my_ner_model")

('./my_ner_model\\tokenizer_config.json',
 './my_ner_model\\special_tokens_map.json',
 './my_ner_model\\tokenizer.json')

In [ ]:
from huggingface_hub import interpreter_login

interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

Enter your token (input will not be visible): ··········
Add token as git credential? (Y/n) n


In [ ]:
#If you have it locally
model = AutoModelForTokenClassification.from_pretrained("./my_ner_model")
tokenizer = AutoTokenizer.from_pretrained("./my_ner_model")

In [ ]:
#Save to HF
model = trainer.model
model.push_to_hub("OOOss/bg-ner-model") #"your-username/your-model-name"
tokenizer.push_to_hub("OOOss/bg-ner-model") #"your-username/your-model-name"

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/OOOss/bg-ner-model/commit/c69a1e7ccf691ec571ededa2c38f72313f058d02', commit_message='Upload tokenizer', commit_description='', oid='c69a1e7ccf691ec571ededa2c38f72313f058d02', pr_url=None, repo_url=RepoUrl('https://huggingface.co/OOOss/bg-ner-model', endpoint='https://huggingface.co', repo_type='model', repo_id='OOOss/bg-ner-model'), pr_revision=None, pr_num=None)

In [ ]:
def free_gpu_memory(model):
    import gc
    del model
    gc.collect()
    torch.cuda.empty_cache()

Build model with Russian data

In [ ]:
language_code="ru"
new_model_name = f"{model_name}-finetuned-{language_code}"

In [ ]:

model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(
    output_dir=f"{language_code}-ner-model",  # Replace with your desired output directory
    learning_rate=3e-5,                       # Common starting point for fine-tuning
    per_device_train_batch_size=16,           # Adjust based on your GPU memory
    per_device_eval_batch_size=16,
    num_train_epochs=5,                       # Increased epochs for better convergence
    weight_decay=0.01,                        # Helps prevent overfitting
    warmup_ratio=0.1,                         # 10% of training steps for warm-up
    lr_scheduler_type="linear",               # Linear learning rate decay
    eval_strategy="epoch",                    # Evaluate at the end of each epoch
    save_strategy="epoch",                    # Save model at the end of each epoch
    logging_steps=700,                        # Log every 100 steps
    save_total_limit=2,                       # Only keep the last 2 checkpoints
    load_best_model_at_end=True,              # Load the best model at the end of training
    metric_for_best_model="f1",               # Use F1 score to evaluate the best model
    seed=42,                                  # For reproducibility
    report_to="wandb"                         # Enable logging to Weights & Biases
)
trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=tokenized_dataset_ru["train"],
                  eval_dataset=tokenized_dataset_ru["validation"],
                  tokenizer=tokenizer)

<ipython-input-9-230d45794dac>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: diko4ev (diko4ev-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.188641,0.760579,0.490172,0.596145,0.967929
2,0.197300,0.195169,0.731984,0.506634,0.598809,0.966966
3,0.197300,0.169206,0.703120,0.548157,0.616043,0.966944
4,0.040900,0.197109,0.722482,0.532187,0.612903,0.967719
5,0.023200,0.206522,0.724469,0.536855,0.616709,0.967763


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=2310, training_loss=0.08067199048541841, metrics={'train_runtime': 3948.478, 'train_samples_per_second': 9.347, 'train_steps_per_second': 0.585, 'total_flos': 8568744350664960.0, 'train_loss': 0.08067199048541841, 'epoch': 5.0})

In [ ]:
#Save to HF
model = trainer.model
model.push_to_hub("OOOss/ru-ner-model") #"your-username/your-model-name"
tokenizer.push_to_hub("OOOss/ru-ner-model") #"your-username/your-model-name"

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/OOOss/ru-not-lem-ner-model/commit/1ced1fc489e9ce1ec761145bf5f85196e383bef7', commit_message='Upload tokenizer', commit_description='', oid='1ced1fc489e9ce1ec761145bf5f85196e383bef7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/OOOss/ru-not-lem-ner-model', endpoint='https://huggingface.co', repo_type='model', repo_id='OOOss/ru-not-lem-ner-model'), pr_revision=None, pr_num=None)

## Models on combined data

Russian + Bulgarian

In [ ]:
tokenized_dataset_bg_ru=concatenate_splits([tokenized_dataset_bg,tokenized_dataset_ru])
print(tokenized_dataset_bg_ru)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 19096
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8149
    })
})


In [ ]:
language_code="bg_ru"
model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(
    output_dir=f"{language_code}-ner-model",  # Replace with your desired output directory
    learning_rate=3e-5,                       # Common starting point for fine-tuning
    per_device_train_batch_size=16,           # Adjust based on your GPU memory
    per_device_eval_batch_size=16,
    num_train_epochs=5,                       # Increased epochs for better convergence
    weight_decay=0.01,                        # Helps prevent overfitting
    warmup_ratio=0.1,                         # 10% of training steps for warm-up
    lr_scheduler_type="linear",               # Linear learning rate decay
    eval_strategy="epoch",              # Evaluate at the end of each epoch
    save_strategy="epoch",                    # Save model at the end of each epoch
    logging_steps=700,                        # Log every 100 steps
    save_total_limit=2,                       # Only keep the last 2 checkpoints
    load_best_model_at_end=True,              # Load the best model at the end of training
    metric_for_best_model="f1",               # Use F1 score to evaluate the best model
    seed=42,                                  # For reproducibility
    report_to="wandb"                         # Enable logging to Weights & Biases
)
trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=tokenized_dataset_bg_ru["train"],
                  eval_dataset=tokenized_dataset_bg_ru["validation"],
                  tokenizer=tokenizer)

<ipython-input-14-8f4d3e38910c>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tretiak-tetiana2 (tretiak-tetiana2-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.263900,0.168212,0.751927,0.593028,0.663091,0.968878
2,0.037200,0.137198,0.763050,0.626132,0.687843,0.970085
3,0.025500,0.160565,0.761670,0.626132,0.687282,0.969375
4,0.017500,0.219571,0.784119,0.607080,0.684335,0.969281
5,0.010700,0.198456,0.785079,0.619916,0.692790,0.970274


TrainOutput(global_step=5970, training_loss=0.05300940456901563, metrics={'train_runtime': 8695.456, 'train_samples_per_second': 10.98, 'train_steps_per_second': 0.687, 'total_flos': 2.216891235879936e+16, 'train_loss': 0.05300940456901563, 'epoch': 5.0})

In [ ]:
#Save to HF
model = trainer.model
model.push_to_hub("tretiakt/bg-ru-ner-model") #"your-username/your-model-name"
tokenizer.push_to_hub("tretiakt/bg-ru-ner-model") #"your-username/your-model-name"

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/tretiakt/bg-ru-ner-model/commit/922c8d2f282d2f7217b4be0dc580368585a41fa3', commit_message='Upload tokenizer', commit_description='', oid='922c8d2f282d2f7217b4be0dc580368585a41fa3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tretiakt/bg-ru-ner-model', endpoint='https://huggingface.co', repo_type='model', repo_id='tretiakt/bg-ru-ner-model'), pr_revision=None, pr_num=None)

All languanges

In [ ]:
all=concatenate_splits([tokenized_dataset_bg,tokenized_dataset_ru, tokenized_dataset_uk, tokenized_dataset_sl])
print(all)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 28819
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15185
    })
})


In [ ]:
language_code="slavic"
model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(
    output_dir=f"{language_code}-ner-model",  # Replace with your desired output directory
    learning_rate=3e-5,                       # Common starting point for fine-tuning
    per_device_train_batch_size=16,           # Adjust based on your GPU memory
    per_device_eval_batch_size=16,
    num_train_epochs=3,                       # Increased epochs for better convergence
    weight_decay=0.01,                        # Helps prevent overfitting
    warmup_ratio=0.1,                         # 10% of training steps for warm-up
    lr_scheduler_type="linear",               # Linear learning rate decay
    eval_strategy="epoch",              # Evaluate at the end of each epoch
    save_strategy="epoch",                    # Save model at the end of each epoch
    logging_steps=700,                        # Log every 100 steps
    save_total_limit=2,                       # Only keep the last 2 checkpoints
    load_best_model_at_end=True,              # Load the best model at the end of training
    metric_for_best_model="f1",               # Use F1 score to evaluate the best model
    seed=42,                                  # For reproducibility
    report_to="wandb"                         # Enable logging to Weights & Biases
)
trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=all["train"],
                  eval_dataset=all["validation"],
                  tokenizer=tokenizer)

<ipython-input-9-833fea7bbc47>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mihaelstoyanov (mihaelstoyanov-it-universitetet-i-k-benhavn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.055100,0.151288,0.715305,0.664201,0.688806,0.969929
2,0.032400,0.136104,0.820208,0.650884,0.725801,0.973848
3,0.019400,0.163993,0.814886,0.682660,0.742936,0.974617


TrainOutput(global_step=5406, training_loss=0.06293657326142611, metrics={'train_runtime': 8078.2422, 'train_samples_per_second': 10.702, 'train_steps_per_second': 0.669, 'total_flos': 2.0073917635156224e+16, 'train_loss': 0.06293657326142611, 'epoch': 3.0})

In [ ]:
#Save to HF
model = trainer.model
model.push_to_hub("mihael199/slavic-ner-model") #"mihael199/your-model-name"
tokenizer.push_to_hub("mihael199/slavic-ner-model") #"mihael199/your-model-name"

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/mihael199/slavic-ner-model/commit/73d09b33c01417bd6924c3eb7ce84388416e9e05', commit_message='Upload tokenizer', commit_description='', oid='73d09b33c01417bd6924c3eb7ce84388416e9e05', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mihael199/slavic-ner-model', endpoint='https://huggingface.co', repo_type='model', repo_id='mihael199/slavic-ner-model'), pr_revision=None, pr_num=None)